# TextThreat Colab Training Notebook

This notebook trains the thesis models for **TextThreat — AI-Powered Detection of Digital Well-Being Risks with Cybersecurity Analytics**.

It trains:

1. SVM + TF-IDF baseline for Jigsaw multi-label toxicity classification.
2. DistilBERT + LoRA improved classifier for Jigsaw multi-label toxicity classification.
3. DistilBERT + LoRA binary stress classifier for Dreaddit.

It also generates the supporting thesis evidence artifacts: sample events, classification metrics, latency, output-level privacy perturbation, fairness demo/audit, and synthetic co-occurrence results.

**Important:** run this notebook in Colab with a GPU runtime for DistilBERT training.

## 1. Select GPU Runtime

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

The SVM baseline can run on CPU, but DistilBERT training is much faster with GPU.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. DistilBERT training will be slow.')

## 2. Clone The Repository

This clones the GitHub repository into the Colab machine. If you forked or moved the repo, change `REPO_URL`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/abdulmuksith3/textthreat-poc.git'
BRANCH = 'next-phase'
# Keep this import close to REPO_DIR so the cell works even when run by itself.
from pathlib import Path
REPO_DIR = Path('/content/textthreat-poc')

if not REPO_DIR.exists():
    !git clone --branch $BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    print('Repository already exists:', REPO_DIR)

os.chdir(REPO_DIR)
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull origin $BRANCH
print('Working directory:', Path.cwd())

## 3. Install Dependencies

This installs the repository requirements, including Transformers, PEFT/LoRA, Fairlearn, Opacus, MLflow, and Splunk/OpenSearch clients.

If Colab asks to restart the runtime after installation, restart and run the notebook again from this point.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. Upload Datasets

Upload these files when prompted:

- `train.csv` from Jigsaw Toxic Comment Classification.
- `dreaddit-train.csv`.
- `dreaddit-test.csv`.

The notebook will place them into the paths expected by the repo:

- `data/jigsaw/train.csv`
- `data/dreaddit/dreaddit-train.csv`
- `data/dreaddit/dreaddit-test.csv`

Raw datasets are used locally in Colab only and should not be committed to Git.

In [ ]:
from google.colab import files
import shutil

Path('data/jigsaw').mkdir(parents=True, exist_ok=True)
Path('data/dreaddit').mkdir(parents=True, exist_ok=True)

print('Upload Jigsaw train.csv and Dreaddit CSV files now.')
uploaded = files.upload()

for name in uploaded.keys():
    lower = name.lower()
    source = Path(name)
    if lower == 'train.csv':
        target = Path('data/jigsaw/train.csv')
    elif 'dreaddit' in lower and 'train' in lower:
        target = Path('data/dreaddit/dreaddit-train.csv')
    elif 'dreaddit' in lower and 'test' in lower:
        target = Path('data/dreaddit/dreaddit-test.csv')
    else:
        print('Leaving unrecognized file in repo root:', name)
        continue
    shutil.move(str(source), target)
    print('Saved', name, '->', target)

## 5. Confirm Dataset Files

This checks that the expected files exist before training starts.

In [ ]:
expected_files = [
    Path('data/jigsaw/train.csv'),
    Path('data/dreaddit/dreaddit-train.csv'),
    Path('data/dreaddit/dreaddit-test.csv'),
]

for path in expected_files:
    print(path, 'exists:', path.exists(), 'size_mb:', round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 'missing')

## 6. Training Configuration

Use `QUICK_TEST = True` first to verify the end-to-end training workflow quickly.

For final thesis evidence, set:

```python
QUICK_TEST = False
JIGSAW_EPOCHS = 1  # or more if you have time/GPU budget
DREADDIT_EPOCHS = 1
```

LoRA is enabled by default because the thesis describes efficient DistilBERT adaptation.

In [ ]:
QUICK_TEST = True

# Set these to True/False if you want to run only part of the training pipeline.
RUN_SVM = True
RUN_JIGSAW_DISTILBERT = True
RUN_DREADDIT_DISTILBERT = True

# LoRA trains small adapter layers instead of the full transformer, reducing memory/time.
USE_LORA = True

# One epoch is enough for a thesis PoC run; increase only if you have GPU time.
JIGSAW_EPOCHS = 1
DREADDIT_EPOCHS = 1

# QUICK_TEST uses small subsets so you can verify that everything works.
SVM_SAMPLE_SIZE = 2000 if QUICK_TEST else None
JIGSAW_SAMPLE_SIZE = 1000 if QUICK_TEST else None
DREADDIT_SAMPLE_SIZE = 1000 if QUICK_TEST else None

print('QUICK_TEST:', QUICK_TEST)
print('USE_LORA:', USE_LORA)

## 7. Smoke Test Before Heavy Training

This verifies the schema, sample exports, metrics, latency, DP, fairness, co-occurrence, and SOAR-lite demo path before spending GPU time.

In [ ]:
!python scripts/smoke_test.py

## 8. Train SVM + TF-IDF Baseline

This is the thesis baseline model. It uses TF-IDF unigrams+bigrams and a calibrated one-vs-rest LinearSVC.

Output:

- `models/svm_tfidf/`
- `experiments/results/svm_metrics.json`

In [ ]:
if RUN_SVM:
    cmd = 'python -m src.textthreat.train_svm --calibration-cv 3'
    if SVM_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {SVM_SAMPLE_SIZE}'
    print('Running:', cmd)
    !$cmd
else:
    print('Skipping SVM baseline training.')

## 9. Train DistilBERT + LoRA On Jigsaw

This is the improved thesis classifier for six-label Jigsaw harm detection.

LoRA configuration in the repo:

- `r = 8`
- `alpha = 16`
- `dropout = 0.1`
- target modules: `q_lin`, `v_lin`

Output:

- `models/distilbert_jigsaw/`
- `experiments/results/distilbert_metrics.json`

In [ ]:
if RUN_JIGSAW_DISTILBERT:
    cmd = f'python -m src.textthreat.train_distilbert --task jigsaw --epochs {JIGSAW_EPOCHS}'
    if JIGSAW_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {JIGSAW_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    print('Running:', cmd)
    !$cmd
else:
    print('Skipping Jigsaw DistilBERT training.')

## 10. Train DistilBERT + LoRA On Dreaddit

This trains the binary stress classifier used by the thesis to support stress signal detection and co-occurrence analytics.

Output:

- `models/distilbert_dreaddit/`
- `experiments/results/dreaddit_metrics.json`

In [ ]:
if RUN_DREADDIT_DISTILBERT:
    cmd = f'python -m src.textthreat.train_distilbert --task dreaddit --epochs {DREADDIT_EPOCHS}'
    if DREADDIT_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {DREADDIT_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    print('Running:', cmd)
    !$cmd
else:
    print('Skipping Dreaddit DistilBERT training.')

## 11. Generate Supporting Thesis Artifacts

These scripts create the evidence JSON files used by the thesis tables and the SIEM/SOAR demo.

The DP script is intentionally described as **output-level privacy-preserving perturbation**, not full DP-SGD training.

In [ ]:
!python -m src.textthreat.export_events --sample
!python -m src.textthreat.evaluate --sample
!python -m src.textthreat.latency --sample
!python -m src.textthreat.dp_output --sample
!python -m src.textthreat.fairness --sample
!python -m src.textthreat.cooccurrence --sample
!python soar_lite/soar_lite.py --demo

## 12. Inspect Result Files

This lists the generated artifacts. These JSON files can be copied into thesis tables or committed when they are demo/sample artifacts.

In [ ]:
from pathlib import Path
import json

for path in sorted(Path('experiments/results').glob('*')):
    print(path, round(path.stat().st_size / 1024, 2), 'KB')

print('\nDistilBERT metrics preview:')
metrics_path = Path('experiments/results/distilbert_metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()) , indent=2)[:2000])
else:
    print('distilbert_metrics.json not found. Check whether training completed.')

## 13. Zip Model And Result Artifacts

This creates a single zip file you can download from Colab. It includes models and result JSONs, but not raw datasets.

In [ ]:
!zip -r textthreat_training_artifacts.zip models experiments/results data/exports/sample_textthreat_events.ndjson schema siem/splunk soar_lite README.md -x '*/__pycache__/*' '*/hf_outputs/*'
print('Created textthreat_training_artifacts.zip')

## 14. Download Artifacts

Download the artifact zip to your machine. You can use the trained model folder later in the local or hosted demo.

In [ ]:
from google.colab import files
files.download('textthreat_training_artifacts.zip')

## 15. Optional: Upload Model To Hugging Face Hub

For the hosted demo, the cleanest deployment is to upload `models/distilbert_jigsaw/` to a Hugging Face model repository and set this environment variable in the app host:

```text
TEXTTHREAT_TOXICITY_MODEL_ID=your-username/textthreat-distilbert-jigsaw
```

The demo can also run with the instant fallback scorer, but trained model upload is better for the final thesis demo.